<a href="https://colab.research.google.com/github/IshiJess/BITACORA-DE-SECUENCIACION--Jessica-Noemi-Salcedo-Alferez-CaBioinf2027/blob/main/bitacora_rnaseq_galaxy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <span style="color:#A230A4;">**Bitácora de RNA seq**</span>

**Estudiante:** Jessica Noemi Salcedo Alferez  
**Fecha:** Octubre 2026  
**Entorno de ejecución:** Galaxy


# Pipeline de Análisis de Expresión Diferencial (RNA-seq) en Galaxy
### Bitácora de Procedimientos y Parámetros

Este cuaderno en google colab funciona como una **bitácora técnica** del análisis de expresión diferencial realizado en la plataforma Galaxy, basado en el protocolo de la **Dra. Alejandra Rougon** (2021/2022).

* **Objetivo:** Conocer y registrar el uso de los programas más utilizados para el análisis de RNA-seq.
* **Datos de entrada:** Datos pareados de Illumina correspondientes a dos líneas celulares de ratón: **G1E** y **Megakaryocytes (MK)** (2 réplicas por condición), mapeados bajo el genoma de referencia **mm10**.

## Etapa 1: Carga e Importación de Datos
Preparación de la historia e importación de lecturas crudas y archivos de anotación de referencia.

In [ ]:
# Parámetros de la herramienta: Get Data -> Upload file -> Paste/Fetch Data
ligas_importacion = [
    'https://zenodo.org/record/583140/files/G1E_rep1_forward_read_%28SRR549355_1%29',
    'https://zenodo.org/record/583140/files/G1E_rep1_reverse_read_%28SRR549355_2%29',
    'https://zenodo.org/record/583140/files/G1E_rep2_forward_read_%28SRR549356_1%29',
    'https://zenodo.org/record/583140/files/G1E_rep2_reverse_read_%28SRR549356_2%29',
    'https://zenodo.org/record/583140/files/Megakaryocyte_rep1_forward_read_%28SRR549357_1%29',
    'https://zenodo.org/record/583140/files/Megakaryocyte_rep1_reverse_read_%28SRR549357_2%29',
    'https://zenodo.org/record/583140/files/Megakaryocyte_rep2_forward_read_%28SRR549358_1%29',
    'https://zenodo.org/record/583140/files/Megakaryocyte_rep2_reverse_read_%28SRR549358_2%29',
    'https://zenodo.org/record/583140/files/RefSeq_reference_GTF_%28DSv2%29'
]

config_formatos = {
    "archivos_lecturas": "fastqsanger",
    "archivo_anotacion": "gtf",
    "genoma_referencia": "Mouse mm10"
}

print("Acción: Modificar nombres de archivos importados para conservar solo la información crítica (ej. G1E_rep1_F).")

## Etapa 2: Control de Calidad y Filtrado de Lecturas
Evaluación de la calidad mediante **FastQC**, agregación con **MultiQC** y remoción de adaptadores/secuencias de baja calidad con **Trimmomatic**.

In [ ]:
# 1. Control de Calidad Inicial
print("Herramienta: FastQC en todos los archivos de lecturas crudas.")

# 2. Agregación de Reportes con MultiQC
multiqc_params = {
    "Which tool was used to generate logs": "FastQC",
    "Type of FastQC output": "Raw data",
    "FastQC output": "Seleccionar todos los archivos Raw data generados"
}

# 3. Limpieza de lecturas con Trimmomatic (repetir para cada par)
trimmomatic_params = {
    "Single-end or paired-end reads?": "Paired-end (two separate input files)",
    "Input FASTQ file (R1)": "G1E_rep1 forward read",
    "Input FASTQ file (R2)": "G1E_rep1 reverse read",
    "Perform initial ILLUMINACLIP step?": "No"
}

print("Nota posterior: Volver a correr FastQC en archivos pareados filtrados para validar la mejora de calidad.")

## Etapa 3: Mapeo a Referencia (Alineamiento con Splicing)
Uso de **HISAT2** para mapear los fragmentos limpios contra el genoma del ratón tomando en cuenta sitios de splicing.

In [ ]:
# Parámetros críticos ejecutados en HISAT2 para todas las muestras:
hisat2_config = {
    "Source for the reference genome": "Use a built-in genome",
    "Select a reference genome": "Mouse (Mus Musculus): mm10",
    "Single-end or paired-end reads?": "Paired-end",
    "FASTA/Q file #1": "Trimmomatic on G1E_rep1 forward read (R1 paired)",
    "FASTA/Q file #2": "Trimmomatic on G1E_rep1 reverse read (R2 paired)",
    "Specify strand information": "Forward(FR)",
    "Advanced options / Spliced alignment options": {
        "Penalty for non-canonical splice sites": '3',
        "Penalty function for long introns with canonical splice sites": 'Constant [f(x) = B]',
        "Constant term (B)": '0.0',
        "Penalty function for long introns with non-canonical splice sites": 'Constant [f(x) = B]'
    },
    "Transcriptome assembly reporting": "Report alignments tailored for transcript assemblers including StringTie"
}

## Etapa 4: Reconstrucción y Ensamble del Transcriptoma
Modelado de transcritos *de novo* utilizando **Stringtie**, unión de ensambles mediante **Stringtie-merge** y evaluación comparativa con **GFFCompare**.

In [ ]:
# 1. Reconstrucción con Stringtie
stringtie_params = {
    "Specify strand information": "Forward (FR)",
    "Action": "Correr sobre todos los alineamientos sin modificar otros parámetros"
}

# 2. Ensamble combinado con Stringtie-merge
stringtie_merge_params = {
    "Transcripts": "Los 4 ensambles generados por Stringtie",
    "Reference annotation to include in the merging": "RefSeqreferenceGTF"
}

# 3. Evaluación con GFFCompare
gffcompare_params = {
    "GTF inputs for comparison": "Output de Stringtie-merge",
    "Use Reference Annotation": "Yes",
    "Choose the source for the reference annotation": "History",
    "Reference Annotation": "RefSeqreferenceGTF",
    "Use Sequence Data": "Yes",
    "Choose the source for the reference list": "Locally cached",
    "Using reference genome": "Mouse (Mus Musculus): mm10"
}

## Etapa 5: Cuantificación (Tabla de Conteos)
Uso de **FeatureCounts** para asignar y contar las lecturas mapeadas sobre los transcritos anotados y combinados.

In [ ]:
# Parámetros aplicados en FeatureCounts
featurecounts_params = {
    "Alignment file": "Los 4 archivos de alineamiento provenientes de HISAT2",
    "Specify strand information": "Stranded (Forward)",
    "Gene annotation file": "Archivo de anotación de transcritos GTF resultante de GFFCompare",
    "Options for paired-end reads": {
        "Count fragments instead of reads": "Enabled (fragments/templates counted instead of single reads)"
    },
    "Advanced options": {
        "GFF gene identifier": "transcript_id"
    }
}

## Etapa 6: Análisis de Expresión Diferencial
Modelado estadístico con **DESeq2** para buscar diferencias significativas de expresión entre las condiciones celulares.

In [ ]:
# Configuración del diseño experimental en DESeq2
deseq2_params = {
    "Factor 1": {
        "Factor level": "G1E",
        "Counts file(s)": "Archivos de featureCounts correspondientes a las 2 réplicas de G1E"
    },
    "Factor 2": {
        "Factor level": "MK",
        "Counts file(s)": "Archivos de featureCounts correspondientes a las 2 réplicas de MK"
    },
    "Visualising the analysis results": "Yes",
    "Output normalized counts table": "Yes"
}

# Filtrado inicial de significancia estadística
print("Herramienta: Filter sobre la salida de DESeq2")
print("Expresión lógica aplicada: c7 < 0.05  (Filtra la columna 7 correspondiente al p-value ajustado < 0.05)")

## Etapa 7: Procesamiento de Tablas y Gráfico de Calor (Heatmap)
Manipulación de datos tabulares a través de comandos de texto y generación del mapa de calor de expresión con `ggplot`.

In [ ]:
# 1. Cruce de datos (Join to Datasets)
join_config = {
    "Join": "La tabla filtrada (P-adj < 0.05)",
    "Using column": "Column 1",
    "with": "La tabla de conteos normalizada de DESeq2",
    "keep the header lines": "no"
}

# 2. Eliminación de columnas sobrantes (cut)
print("Herramienta: cut -> operation: discard -> fields: column 1,2,3,4,5,6,7")

# 3. Inserción de un marcador de encabezado
print("Herramienta: Add line to file -> text to add: ADD_HEADER")

# 4. Formateo y corrección de encabezados con sed
sed_program = r"s/ADD_HEADER/ID\tMK_1\tMK_2\tG1E_1\tG1E_2/"
print(f"Herramienta: Text transformation with sed -> Programa SED: {sed_program}")

# 5. Configuración de visualización con heatmap w ggplot
heatmap_ggplot_params = {
    "Select input dataset options": "Dataset with header and row names",
    "Select column for row names": "column 1",
    "Samples name orientation": "vertical",
    "Data transformation": "Log2 value",
    "Heatmap colour scheme": "Blue to white to red"
}
print("Acción: Descargar mapa de calor visualizado.")

## Etapa 8: Dirección de Regulación de Transcritos
Segregación final de los transcritos significativos en sobre-expresados (Up-regulated) y sub-expresados (Down-regulated) en la línea celular **G1E**.

In [ ]:
print("Filtro aplicado sobre la tabla de transcritos significativos obtenidos en la Etapa 6 (c7 < 0.05):")

filtros_regulacion = {
    "Transcritos Sobre-regulados (Up-regulated) en G1E": "c3 > 0  (Columna 3 log2FoldChange mayor a 0)",
    "Transcritos Sub-regulados (Down-regulated) en G1E": "c3 < 0  (Columna 3 log2FoldChange menor a 0)"
}

for tipo, condicion in filtros_regulacion.items():
    print(f"* {tipo} -> Expresión: {condicion}")